# Improved Vision Transformer for Stock Return Prediction

**Authors:** Tejas Singh, Ikenna Odikamnoro, Navya Sehgal  
**Project:** UBC Sauder COMM 486 | Supervised by Prof. Jan Bena, Prof. Jose Pizarro Zuniga, Prof. James Shou  
**Based on:** Byun, Na, and Song (2025) — From Vision to Value: Stock Chart Image-Driven Factors and Their Pricing Power

---

## What is different from the baseline model?

The baseline model (Ikenna's version) used:
- ViT-Base/32 with 2 encoder layers trained from scratch
- Adam optimizer with fixed learning rate
- No image augmentation
- Standard cross-entropy loss

This improved version adds:
1. **4 encoder layers** — more depth gives the model more capacity to learn complex chart patterns
2. **AdamW optimizer** — a better version of Adam that handles weight decay more correctly
3. **Cosine learning rate scheduler** — the learning rate gradually decreases following a cosine curve, which leads to smoother and more stable convergence
4. **Image augmentation** — random horizontal flips and slight colour jitter reduce overfitting by showing the model slightly varied versions of each image
5. **Label smoothing** — instead of training the model to predict exactly 0 or 1, we use 0.9 and 0.1. This improves probability calibration, which directly addresses the quintile ranking issue observed in our results
6. **Mixed precision training** — uses 16-bit floats where possible to speed up training on GPU without losing accuracy
7. **Gradient clipping** — prevents the model weights from updating too aggressively in a single step, which stabilises training


In [ ]:
# ─── CELL 1: Check GPU availability ──────────────────────────────────────────
# Before doing anything, confirm that a GPU is available.
# Training on CPU would take weeks — GPU is required for this scale of data.

import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("WARNING: No GPU found. Training will be very slow.")

In [ ]:
# ─── CELL 2: Install and import all required libraries ────────────────────────
# All libraries below come pre-installed on SageMaker and most GPU environments.
# If running locally, install with: pip install torch torchvision boto3

import os
import zipfile
import json
import boto3
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast   # mixed precision training
from torchvision.models import vit_b_32
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split

# Set the device — use GPU if available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ─── CELL 3: Set all configuration values in one place ───────────────────────
# Keeping all settings here makes it easy to change and experiment.
# These values follow the Byun et al. paper specification where applicable.

CONFIG = {
    # Data paths — update these to match where your images are stored
    "bucket":           "vit-stock-data-ikenna14",           # S3 bucket name
    "s3_train_up":      "vit-data/train/up/Train_up.zip",    # S3 path to UP images
    "s3_train_down":    "vit-data/train/down/Train_down.zip",# S3 path to DOWN images
    "base_dir":         "/mnt/sagemaker-nvme/vit_data",      # local storage directory

    # Model architecture
    "num_encoder_layers": 4,     # IMPROVEMENT: 4 layers instead of 2 — more capacity to learn patterns
    "num_classes":        2,     # binary classification: up or down
    "image_size":         224,   # must match the input size of ViT-Base/32

    # Training settings
    "batch_size":         32,
    "num_epochs":         10,
    "learning_rate":      1e-4,
    "weight_decay":       1e-4,
    "label_smoothing":    0.1,   # IMPROVEMENT: instead of hard 0/1, use 0.1/0.9 for better probability calibration
    "grad_clip":          1.0,   # IMPROVEMENT: clip gradients to prevent unstable weight updates
    "train_split":        0.7,   # 70% of data for training, 30% for validation

    # Save paths
    "best_model_path":   "best_improved_vit.pth",
    "checkpoint_path":   "improved_checkpoint.pth",
    "history_path":      "training_history.json",
}

print("Configuration loaded.")
print(f"Encoder layers: {CONFIG['num_encoder_layers']} (baseline was 2)")
print(f"Label smoothing: {CONFIG['label_smoothing']} (baseline was 0.0)")
print(f"Gradient clipping: {CONFIG['grad_clip']} (baseline had none)")

In [ ]:
# ─── CELL 4: Download image data from S3 ─────────────────────────────────────
# Images are stored in AWS S3 as zip files.
# We download them once and skip the download if they already exist locally.
# This saves time when restarting training after an interruption.

s3 = boto3.client("s3")
BASE_DIR = CONFIG["base_dir"]

os.makedirs(f"{BASE_DIR}/train/up", exist_ok=True)
os.makedirs(f"{BASE_DIR}/train/down", exist_ok=True)

train_up_path   = f"{BASE_DIR}/train_up.zip"
train_down_path = f"{BASE_DIR}/train_down.zip"

# Download UP images if not already downloaded
if not os.path.exists(train_up_path):
    print("Downloading UP images from S3...")
    s3.download_file(CONFIG["bucket"], CONFIG["s3_train_up"], train_up_path)
    print("Done.")
else:
    print("UP images already downloaded, skipping.")

# Download DOWN images if not already downloaded
if not os.path.exists(train_down_path):
    print("Downloading DOWN images from S3...")
    s3.download_file(CONFIG["bucket"], CONFIG["s3_train_down"], train_down_path)
    print("Done.")
else:
    print("DOWN images already downloaded, skipping.")

In [ ]:
# ─── CELL 5: Unzip images into the correct folder structure ──────────────────
# ImageFolder (used in Cell 7) requires this exact structure:
#   train/
#     up/     <- images where stock went UP after the chart
#     down/   <- images where stock went DOWN after the chart

up_extract_path   = f"{BASE_DIR}/train/up"
down_extract_path = f"{BASE_DIR}/train/down"

# Only unzip if the folders are empty — avoids re-extracting unnecessarily
if len(os.listdir(up_extract_path)) == 0:
    print("Extracting UP images...")
    with zipfile.ZipFile(train_up_path, 'r') as z:
        z.extractall(up_extract_path)
    print("Done.")
else:
    print(f"UP folder already has {len(os.listdir(up_extract_path))} images, skipping.")

if len(os.listdir(down_extract_path)) == 0:
    print("Extracting DOWN images...")
    with zipfile.ZipFile(train_down_path, 'r') as z:
        z.extractall(down_extract_path)
    print("Done.")
else:
    print(f"DOWN folder already has {len(os.listdir(down_extract_path))} images, skipping.")

print(f"\nTotal UP images:   {len(os.listdir(up_extract_path))}")
print(f"Total DOWN images: {len(os.listdir(down_extract_path))}")

In [ ]:
# ─── CELL 6: Define image transforms ─────────────────────────────────────────
# Transforms are applied to every image before it is fed into the model.
#
# TRAINING transforms include augmentation:
#   - RandomHorizontalFlip: randomly mirror the chart left-to-right.
#     This helps the model generalise — a downtrend mirrored is not a different pattern.
#   - ColorJitter: slightly vary brightness and contrast.
#     This prevents the model from over-relying on exact colour values.
#
# VALIDATION transforms do NOT include augmentation.
#   We evaluate on the original unmodified images so the validation score is reliable.
#
# Both use Normalize with ImageNet mean and std values.
# These are standard values used when working with pretrained vision models.
# Even though we train from scratch, using these values keeps pixel values
# in a sensible numerical range for the model to work with.

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.RandomHorizontalFlip(p=0.3),             # IMPROVEMENT: random flip for augmentation
    transforms.ColorJitter(brightness=0.1, contrast=0.1),  # IMPROVEMENT: slight colour variation
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),  # IMPROVEMENT: proper normalisation
])

val_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Train transform (with augmentation) and validation transform defined.")

In [ ]:
# ─── CELL 7: Load dataset and split into train and validation ─────────────────
# ImageFolder automatically reads the folder structure and assigns labels:
#   down/ -> class 0
#   up/   -> class 1
#
# We use a 70/30 split: 70% of images for training, 30% for validation.
# This matches Ikenna's baseline split.
#
# NOTE: We apply different transforms to train and validation sets.
# To do this cleanly we load the dataset twice with different transforms.

from torch.utils.data import Subset

# Load the full dataset first to get the split indices
full_dataset_for_split = datasets.ImageFolder(
    root=f"{BASE_DIR}/train",
    transform=train_transform
)

full_size  = len(full_dataset_for_split)
train_size = int(CONFIG["train_split"] * full_size)
val_size   = full_size - train_size

# Create a random split — same seed every time so results are reproducible
generator = torch.Generator().manual_seed(42)
train_indices, val_indices = random_split(
    range(full_size), [train_size, val_size], generator=generator
)

# Load dataset again with val transform for validation subset
full_dataset_val = datasets.ImageFolder(
    root=f"{BASE_DIR}/train",
    transform=val_transform
)

train_dataset = Subset(full_dataset_for_split, train_indices)
val_dataset   = Subset(full_dataset_val, val_indices)

print(f"Classes: {full_dataset_for_split.classes}")
print(f"Total images: {full_size:,}")
print(f"Training set: {len(train_dataset):,} images")
print(f"Validation set: {len(val_dataset):,} images")

In [ ]:
# ─── CELL 8: Create data loaders ──────────────────────────────────────────────
# Data loaders handle batching, shuffling, and loading images efficiently.
#
# num_workers: number of CPU threads loading images in the background.
#   More workers = faster data loading. Set based on how many CPU cores are available.
#
# pin_memory=True: keeps loaded data in a special memory area for faster GPU transfer.
#   Always use this when training on GPU.
#
# persistent_workers=True: keeps worker processes alive between epochs.
#   This avoids the overhead of starting new workers every epoch.

NUM_WORKERS = min(4, os.cpu_count())  # use up to 4 CPU threads for loading

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,              # shuffle training data every epoch
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,   # IMPROVEMENT: keep workers alive between epochs
    drop_last=True,            # drop the last incomplete batch to avoid training instability
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,             # no shuffling for validation — order does not matter
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

print(f"Train loader: {len(train_loader):,} batches of {CONFIG['batch_size']} images")
print(f"Val loader:   {len(val_loader):,} batches of {CONFIG['batch_size']} images")

In [ ]:
# ─── CELL 9: Build the improved ViT model ────────────────────────────────────
# We start from the standard ViT-Base/32 architecture (no pretrained weights).
#
# Key changes from baseline:
#   1. num_encoder_layers=4 instead of 2
#      More layers means the model can learn more complex, multi-level visual patterns.
#      Each transformer layer refines the representation from the previous one.
#      The paper used 2 layers due to compute constraints. We use 4 for better capacity.
#
#   2. Stochastic Depth (drop_path) regularisation
#      During training, each transformer layer is randomly skipped with a small probability.
#      This prevents the model from becoming too dependent on any single layer
#      and acts as a form of regularisation that improves generalisation.
#
#   3. The final classification head is replaced with a 2-class linear layer.
#      ViT-Base/32 originally outputs 1000 classes (ImageNet). We change this to 2.

def create_improved_vit(num_encoder_layers=4, num_classes=2, drop_path_rate=0.1):
    """
    Creates an improved ViT-Base/32 model for binary stock return classification.
    
    Args:
        num_encoder_layers: number of transformer encoder layers (default 4, baseline used 2)
        num_classes: number of output classes (2 for up/down)
        drop_path_rate: probability of dropping a transformer layer during training
    
    Returns:
        PyTorch model ready for training
    """
    # Load the standard ViT-Base/32 architecture without any pretrained weights
    model = vit_b_32(weights=None)

    # Keep only the first num_encoder_layers transformer blocks
    model.encoder.layers = nn.Sequential(
        *[model.encoder.layers[i] for i in range(num_encoder_layers)]
    )

    # Add stochastic depth (drop path) to each encoder layer for regularisation
    # Each layer gets a slightly higher drop probability than the previous one
    drop_probs = [drop_path_rate * i / num_encoder_layers for i in range(num_encoder_layers)]
    for i, layer in enumerate(model.encoder.layers):
        if hasattr(layer, 'drop_path'):
            layer.drop_path.p = drop_probs[i]

    # Replace the 1000-class ImageNet head with a 2-class head for up/down prediction
    in_features = model.heads.head.in_features
    model.heads.head = nn.Sequential(
        nn.LayerNorm(in_features),        # normalise before classification
        nn.Dropout(p=0.1),                # IMPROVEMENT: dropout for regularisation
        nn.Linear(in_features, num_classes)
    )

    return model


# Build the model and move it to GPU
model = create_improved_vit(
    num_encoder_layers=CONFIG["num_encoder_layers"],
    num_classes=CONFIG["num_classes"]
)
model = model.to(device)

# Count total trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready on: {device}")
print(f"Total trainable parameters: {total_params:,}")
print(f"Encoder layers: {CONFIG['num_encoder_layers']} (baseline was 2)")

In [ ]:
# ─── CELL 10: Set up loss function, optimizer, and learning rate scheduler ───
#
# Loss function: CrossEntropyLoss with label_smoothing=0.1
#   Standard cross-entropy trains the model to output probabilities as close to
#   exactly 0 or 1 as possible. With label smoothing, we train it to output
#   probabilities close to 0.1 and 0.9 instead.
#   WHY: This directly improves probability calibration — the model's confidence
#   scores better reflect actual likelihood. This is the key fix for the non-monotonic
#   quintile ranking issue observed in the CC&L presentation.
#
# Optimizer: AdamW
#   AdamW is a corrected version of Adam that applies weight decay properly.
#   In the original Adam, weight decay interacts with the adaptive learning rate
#   in a way that weakens its regularisation effect. AdamW fixes this.
#
# Learning rate scheduler: CosineAnnealingLR
#   Instead of keeping the learning rate fixed, it starts at the initial value
#   and gradually decreases following a cosine curve down to near zero.
#   WHY: A high learning rate early in training helps the model explore broadly.
#   A low learning rate late in training helps it settle into a good solution.
#   This typically produces better final accuracy than a fixed learning rate.

criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])

optimizer = optim.AdamW(                          # IMPROVEMENT: AdamW instead of Adam
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)

scheduler = optim.lr_scheduler.CosineAnnealingLR( # IMPROVEMENT: cosine learning rate schedule
    optimizer,
    T_max=CONFIG["num_epochs"],
    eta_min=1e-6                                   # minimum learning rate at end of schedule
)

# Mixed precision scaler — handles the scaling needed for 16-bit float training
# This speeds up training significantly on modern GPUs with no loss in accuracy
scaler = GradScaler()                              # IMPROVEMENT: mixed precision training

print("Loss function: CrossEntropyLoss with label_smoothing =", CONFIG["label_smoothing"])
print("Optimizer: AdamW | lr =", CONFIG["learning_rate"], "| weight_decay =", CONFIG["weight_decay"])
print("Scheduler: CosineAnnealingLR over", CONFIG["num_epochs"], "epochs")
print("Mixed precision training: enabled")

In [ ]:
# ─── CELL 11: Define training and validation functions ────────────────────────
#
# train_one_epoch: runs through the full training set once
#   - Uses mixed precision (autocast) for faster GPU computation
#   - Clips gradients to prevent very large weight updates
#   - Prints progress every 1000 batches so you can monitor training
#
# validate_one_epoch: runs through the full validation set once
#   - torch.no_grad() disables gradient computation (not needed during validation)
#   - This saves memory and speeds up the validation pass
#   - model.eval() switches off dropout and batch norm during validation

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, grad_clip):
    """
    Runs one full pass through the training data.
    Returns average loss and accuracy for the epoch.
    """
    model.train()       # set model to training mode (enables dropout)
    running_loss = 0.0
    correct      = 0
    total        = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()  # clear gradients from the previous batch

        # autocast runs the forward pass in 16-bit float for speed
        with autocast():       # IMPROVEMENT: mixed precision forward pass
            outputs = model(images)
            loss    = criterion(outputs, labels)

        # scaler handles the backward pass in a numerically stable way
        scaler.scale(loss).backward()

        # Clip gradients — prevents any single weight from updating too aggressively
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)  # IMPROVEMENT

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        preds    = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

        if batch_idx % 1000 == 0:
            print(f"  Train Batch {batch_idx:>5} | Loss: {loss.item():.4f} | "
                  f"Running Acc: {correct / total:.4f}")

    return running_loss / total, correct / total


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    """
    Runs one full pass through the validation data.
    Returns average loss and accuracy for the epoch.
    """
    model.eval()        # set model to evaluation mode (disables dropout)
    running_loss = 0.0
    correct      = 0
    total        = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast():   # mixed precision in validation too for speed
            outputs = model(images)
            loss    = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds    = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    return running_loss / total, correct / total


print("Training and validation functions defined.")

In [ ]:
# ─── CELL 12: Load checkpoint if resuming interrupted training ────────────────
# If training was interrupted (Colab timeout, SageMaker shutdown, etc.),
# this cell loads the last saved checkpoint so we continue from where we stopped.
# If no checkpoint exists, we start from scratch.

start_epoch  = 0
best_val_acc = 0.0
history      = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}

if os.path.exists(CONFIG["checkpoint_path"]):
    print("Checkpoint found — resuming training...")
    checkpoint = torch.load(CONFIG["checkpoint_path"], map_location=device)

    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    scaler.load_state_dict(checkpoint["scaler_state"])

    start_epoch  = checkpoint["epoch"] + 1
    best_val_acc = checkpoint["best_val_acc"]

    if os.path.exists(CONFIG["history_path"]):
        with open(CONFIG["history_path"], "r") as f:
            history = json.load(f)

    print(f"Resuming from epoch {start_epoch} | Best val acc so far: {best_val_acc:.4f}")
else:
    print("No checkpoint found — starting training from scratch.")

In [ ]:
# ─── CELL 13: Main training loop ─────────────────────────────────────────────
# This is the main loop that trains the model for the configured number of epochs.
#
# Each epoch:
#   1. Trains on all training images
#   2. Evaluates on all validation images
#   3. Steps the learning rate scheduler (decreases lr following cosine curve)
#   4. Saves the model if validation accuracy improved
#   5. Saves a checkpoint so training can be resumed if interrupted
#   6. Logs all metrics to a JSON history file

print("="*70)
print("STARTING IMPROVED ViT TRAINING")
print(f"Epochs: {start_epoch} to {CONFIG['num_epochs']}")
print(f"Encoder layers: {CONFIG['num_encoder_layers']}")
print(f"Label smoothing: {CONFIG['label_smoothing']}")
print(f"Gradient clipping: {CONFIG['grad_clip']}")
print("="*70)

for epoch in range(start_epoch, CONFIG["num_epochs"]):

    current_lr = scheduler.get_last_lr()[0]
    print(f"\nEpoch [{epoch+1}/{CONFIG['num_epochs']}] | Learning rate: {current_lr:.2e}")
    print("-" * 50)

    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device, CONFIG["grad_clip"]
    )

    # Validate
    val_loss, val_acc = validate_one_epoch(
        model, val_loader, criterion, device
    )

    # Step the learning rate scheduler
    scheduler.step()  # IMPROVEMENT: cosine lr decay applied each epoch

    # Print epoch summary
    print(
        f"Epoch [{epoch+1}/{CONFIG['num_epochs']}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    # Save best model if validation accuracy improved
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CONFIG["best_model_path"])
        print(f"  --> New best model saved (val acc: {best_val_acc:.4f})")

    # Save checkpoint every epoch (enables training resumption)
    torch.save({
        "epoch":           epoch,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state":    scaler.state_dict(),
        "best_val_acc":    best_val_acc,
    }, CONFIG["checkpoint_path"])

    # Log training history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    with open(CONFIG["history_path"], "w") as f:
        json.dump(history, f, indent=2)

print("\n" + "="*70)
print(f"Training complete. Best validation accuracy: {best_val_acc:.4f}")
print("="*70)

In [ ]:
# ─── CELL 14: Plot training history ──────────────────────────────────────────
# Visualise how loss and accuracy changed over training.
# A good training run shows:
#   - Loss decreasing over epochs
#   - Accuracy increasing over epochs
#   - Train and validation curves staying close together (no large gap = no overfitting)

import matplotlib.pyplot as plt

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Improved ViT Training History", fontsize=14, fontweight="bold")

# Loss plot
axes[0].plot(epochs_range, history["train_loss"], label="Train Loss", color="#CC0000")
axes[0].plot(epochs_range, history["val_loss"],   label="Val Loss",   color="#333333", linestyle="--")
axes[0].set_title("Loss per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy plot
axes[1].plot(epochs_range, history["train_acc"], label="Train Acc", color="#CC0000")
axes[1].plot(epochs_range, history["val_acc"],   label="Val Acc",   color="#333333", linestyle="--")
axes[1].set_title("Accuracy per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

# Learning rate plot
axes[2].plot(epochs_range, history["lr"], color="#CC0000")
axes[2].set_title("Learning Rate Schedule (Cosine)")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Learning Rate")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_history.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved as training_history.png")

In [ ]:
# ─── CELL 15: Run inference and generate probability scores ───────────────────
# After training, we load the best saved model and run it on a batch of images.
# The output is a probability score for each image: P(up) in [0, 1].
#
# This is the signal that feeds into portfolio construction.
# Higher P(up) means the model is more confident the stock will go up.
# Stocks are ranked by this score and sorted into quintiles for the long-short strategy.
#
# With label smoothing and better calibration, these probability scores should
# rank stocks more cleanly than the baseline model, producing a monotonic
# relationship between quintile rank and actual returns.

import torch.nn.functional as F

# Load the best model weights
print("Loading best model...")
model.load_state_dict(torch.load(CONFIG["best_model_path"], map_location=device))
model.eval()
print("Model loaded.")

# Run inference on a sample batch to verify output format
sample_images, sample_labels = next(iter(val_loader))
sample_images = sample_images.to(device)

with torch.no_grad():
    with autocast():
        logits = model(sample_images)                     # raw model output
        probs  = F.softmax(logits, dim=1)                 # convert to probabilities
        prob_up = probs[:, 1].cpu().numpy()               # P(up) for each image

print(f"\nSample batch size: {len(prob_up)} images")
print(f"P(up) scores (first 10): {prob_up[:10].round(3)}")
print(f"Score range: {prob_up.min():.3f} to {prob_up.max():.3f}")
print("\nThese probability scores are the signal for portfolio construction.")
print("Sort all stocks by P(up) at month end → long top quintile, short bottom quintile.")

In [ ]:
# ─── CELL 16: Summary of all improvements ────────────────────────────────────
# A clear comparison between the baseline model and this improved version.

print("=" * 65)
print("IMPROVEMENT SUMMARY")
print("=" * 65)

improvements = [
    ("Encoder layers",       "2",              "4",
     "More capacity to learn complex chart patterns"),
    ("Optimizer",            "Adam",           "AdamW",
     "Correct weight decay handling"),
    ("LR schedule",          "Fixed 1e-4",     "Cosine decay 1e-4 to 1e-6",
     "Better convergence, avoids getting stuck"),
    ("Label smoothing",      "None (0.0)",     "0.1",
     "Better probability calibration, fixes quintile ranking"),
    ("Image augmentation",   "None",           "Flip + ColorJitter",
     "Reduces overfitting"),
    ("Mixed precision",      "No",             "Yes (FP16)",
     "Faster training, same accuracy"),
    ("Gradient clipping",    "No",             "Yes (max norm 1.0)",
     "Stable training, prevents exploding gradients"),
    ("Classification head",  "Linear only",   "LayerNorm + Dropout + Linear",
     "Better regularisation before prediction"),
    ("Image normalisation",  "ToTensor only", "ImageNet mean/std normalise",
     "Correct input scaling for the model"),
]

print(f"{'Feature':<22} {'Baseline':<22} {'Improved':<28} {'Why'}")
print("-" * 65)
for feat, base, improved, reason in improvements:
    print(f"{feat:<22} {base:<22} {improved:<28} {reason}")

print("\nKey motivation:")
print("Label smoothing is the most important change for portfolio performance.")
print("It directly addresses the non-monotonic quintile ranking observed in results,")
print("producing better-calibrated P(up) scores for cross-sectional stock ranking.")